In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

class ImprovedForecastingSystem:
    """개선된 통합 예측 시스템"""
    
    def __init__(self, seq_length=28, pred_length=7):
        self.seq_length = seq_length
        self.pred_length = pred_length
        
        # 업장별 특성 정의 (단순화)
        self.store_config = {
            '담하': {'type': 'premium', 'baseline': 8.0, 'volatility': 'medium'},
            '미라시아': {'type': 'brunch', 'baseline': 6.0, 'volatility': 'high'},
            '포레스트릿': {'type': 'cafe', 'baseline': 48.0, 'volatility': 'low'},
            '카페테리아': {'type': 'restaurant', 'baseline': 19.0, 'volatility': 'low'},
            '화담숲주막': {'type': 'traditional', 'baseline': 34.0, 'volatility': 'medium'},
            '화담숲카페': {'type': 'cafe', 'baseline': 24.0, 'volatility': 'medium'},
            '느티나무 셀프BBQ': {'type': 'bbq', 'baseline': 6.0, 'volatility': 'high'},
            '연회장': {'type': 'event', 'baseline': 2.0, 'volatility': 'very_high'},
            '라그로타': {'type': 'fine_dining', 'baseline': 1.0, 'volatility': 'very_high'}
        }
        
        # 통합 모델 (업장별 분리 대신)
        self.main_model = None
        self.zero_classifier = None
        self.scalers = {}
        self.encoders = {}
        
    def preprocess_data(self, df):
        """개선된 데이터 전처리"""
        df = df.copy()
        
        # 1. 기본 시간 피처
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day'] = df['date'].dt.day
        df['day_of_week'] = df['date'].dt.dayofweek
        df['week_of_year'] = df['date'].dt.isocalendar().week
        df['day_of_year'] = df['date'].dt.dayofyear
        
        # 2. 주기적 인코딩 (사인/코사인)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        
        # 3. 휴일 및 특별일 처리
        df['is_weekend'] = df['day_of_week'].isin([5, 6])
        df['is_friday'] = (df['day_of_week'] == 4)
        df['is_monday'] = (df['day_of_week'] == 0)
        
        # 4. 계절 및 성수기
        df['season'] = ((df['month'] % 12 + 3) // 3).map({1: 'spring', 2: 'summer', 3: 'fall', 4: 'winter'})
        df['is_peak_season'] = df['month'].isin([7, 8, 12, 1])
        df['is_summer'] = df['month'].isin([6, 7, 8])
        df['is_winter'] = df['month'].isin([12, 1, 2])
        
        # 5. 업장 타입 매핑
        df['store_type'] = df['store'].map(lambda x: self.store_config.get(x, {}).get('type', 'other'))
        df['store_baseline'] = df['store'].map(lambda x: self.store_config.get(x, {}).get('baseline', 5.0))
        
        return df
    
    def create_sequences(self, df, mode='train'):
        """시계열 시퀀스 생성"""
        sequences = []
        targets = []
        metadata = []
        
        # 업장-메뉴별로 시퀀스 생성
        for (store, menu), group in df.groupby(['store', 'menu']):
            group = group.sort_values('date').reset_index(drop=True)
            
            min_length = self.seq_length + (self.pred_length if mode == 'train' else 0)
            if len(group) < min_length:
                continue
            
            if mode == 'predict':
                # 예측 모드: 마지막 28일만 사용
                seq_data = group.tail(self.seq_length)
                X, meta = self._create_sequence_features(seq_data, store, menu, mode)
                sequences.append(X)
                metadata.append(meta)
                
            else:
                # 학습 모드: 슬라이딩 윈도우
                for i in range(len(group) - min_length + 1):
                    seq_data = group.iloc[i:i+self.seq_length]
                    target_data = group.iloc[i+self.seq_length:i+self.seq_length+self.pred_length]
                    
                    X, meta = self._create_sequence_features(seq_data, store, menu, mode)
                    y = target_data['sales'].values
                    
                    sequences.append(X)
                    targets.append(y)
                    metadata.append(meta)
        
        X = np.array(sequences) if sequences else np.empty((0, self._get_feature_size()))
        y = np.array(targets) if targets else np.empty((0, self.pred_length))
        
        return X, y, metadata
    
    def _create_sequence_features(self, seq_data, store, menu, mode='train'):
        """단일 시퀀스 피처 생성"""
        
        # 1. 매출 통계 피처
        sales = seq_data['sales'].values
        
        features = {
            # 기본 통계
            'sales_mean': np.mean(sales),
            'sales_std': np.std(sales),
            'sales_median': np.median(sales),
            'sales_max': np.max(sales),
            'sales_min': np.min(sales),
            
            # 최근 패턴
            'sales_last_1': sales[-1],
            'sales_last_3_mean': np.mean(sales[-3:]),
            'sales_last_7_mean': np.mean(sales[-7:]),
            'sales_last_14_mean': np.mean(sales[-14:]),
            
            # 트렌드
            'sales_trend': self._calculate_trend(sales),
            'sales_momentum': np.mean(sales[-7:]) - np.mean(sales[-14:-7]) if len(sales) >= 14 else 0,
            
            # 0 매출 관련
            'zero_ratio': np.mean(sales == 0),
            'zero_streak': self._max_consecutive_zeros(sales),
            'days_since_last_sale': self._days_since_last_sale(sales),
            
            # 요일별 패턴
            'weekday_mean': seq_data[~seq_data['is_weekend']]['sales'].mean(),
            'weekend_mean': seq_data[seq_data['is_weekend']]['sales'].mean(),
            'friday_mean': seq_data[seq_data['is_friday']]['sales'].mean(),
            
            # 업장 특성
            'store_baseline': self.store_config.get(store, {}).get('baseline', 5.0),
            'relative_performance': np.mean(sales) / self.store_config.get(store, {}).get('baseline', 5.0),
            
            # 시간 정보
            'current_month': seq_data['month'].iloc[-1],
            'current_dow': seq_data['day_of_week'].iloc[-1],
            'is_peak_season': seq_data['is_peak_season'].iloc[-1],
            'is_summer': seq_data['is_summer'].iloc[-1],
            
            # 순환 피처
            'month_sin': seq_data['month_sin'].iloc[-1],
            'month_cos': seq_data['month_cos'].iloc[-1],
            'dow_sin': seq_data['dow_sin'].iloc[-1],
            'dow_cos': seq_data['dow_cos'].iloc[-1],
        }
        
        # 2. 메뉴 카테고리 (단순화)
        menu_lower = str(menu).lower()
        features.update({
            'is_main_dish': int(any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥'])),
            'is_drink': int(any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피', '음료'])),
            'is_dessert': int(any(x in menu_lower for x in ['아이스', '케이크', '디저트'])),
            'is_set_menu': int(any(x in menu_lower for x in ['정식', '세트', '패키지'])),
            'is_brunch': int('브런치' in menu_lower),
        })
        
        # 3. 업장 타입
        store_type = self.store_config.get(store, {}).get('type', 'other')
        for stype in ['premium', 'brunch', 'cafe', 'restaurant', 'traditional', 'bbq', 'event', 'fine_dining']:
            features[f'store_type_{stype}'] = int(store_type == stype)
        
        # NaN 처리
        for key, value in features.items():
            if pd.isna(value) or np.isinf(value):
                features[key] = 0.0
        
        feature_vector = np.array(list(features.values()))
        metadata = {'store': store, 'menu': menu, 'seq_end_date': seq_data['date'].iloc[-1]}
        
        return feature_vector, metadata
    
    def _calculate_trend(self, sales):
        """선형 트렌드 계산"""
        if len(sales) < 2:
            return 0
        x = np.arange(len(sales))
        return np.polyfit(x, sales, 1)[0]
    
    def _max_consecutive_zeros(self, sales):
        """최대 연속 0 개수"""
        max_zeros = current_zeros = 0
        for sale in sales:
            if sale == 0:
                current_zeros += 1
                max_zeros = max(max_zeros, current_zeros)
            else:
                current_zeros = 0
        return max_zeros
    
    def _days_since_last_sale(self, sales):
        """마지막 매출 이후 일수"""
        for i in range(len(sales)-1, -1, -1):
            if sales[i] > 0:
                return len(sales) - 1 - i
        return len(sales)
    
    def _get_feature_size(self):
        """피처 크기 계산"""
        return 35  # 대략적인 피처 수
    
    def fit(self, train_df):
        """모델 학습"""
        print("개선된 모델 학습 시작...")
        
        # 전처리
        train_processed = self.preprocess_data(train_df)
        
        # 시퀀스 생성
        X, y, metadata = self.create_sequences(train_processed, mode='train')
        
        if len(X) == 0:
            print("학습 데이터 부족")
            return
        
        print(f"학습 시퀀스: {X.shape}, 타겟: {y.shape}")
        
        # 1. Zero-inflation 분류기 학습
        has_sales = (y > 0).any(axis=1)
        
        from sklearn.ensemble import RandomForestClassifier
        self.zero_classifier = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            class_weight='balanced'
        )
        self.zero_classifier.fit(X, has_sales)
        
        # 2. 매출량 예측 모델 학습 (멀티아웃풋)
        sales_mask = has_sales
        if sales_mask.sum() > 10:
            from sklearn.multioutput import MultiOutputRegressor
            self.main_model = MultiOutputRegressor(
                LGBMRegressor(
                    n_estimators=500,
                    max_depth=8,
                    learning_rate=0.05,
                    feature_fraction=0.8,
                    bagging_fraction=0.8,
                    random_state=42,
                    verbosity=-1
                )
            )
            
            # 가중치 적용 (0에 가까운 값은 낮은 가중치)
            sample_weights = np.where(y[sales_mask].mean(axis=1) < 1, 0.5, 1.0)
            
            # fit 메서드가 sample_weight를 지원하는지 확인 후 사용
            try:
                self.main_model.fit(X[sales_mask], y[sales_mask], sample_weight=sample_weights)
            except:
                self.main_model.fit(X[sales_mask], y[sales_mask])
        
        print("모델 학습 완료")
    
    def predict(self, test_df):
        """예측"""
        if self.main_model is None:
            print("모델이 학습되지 않음")
            return np.zeros((len(test_df), self.pred_length))
        
        # 전처리
        test_processed = self.preprocess_data(test_df)
        
        # 시퀀스 생성
        X, _, metadata = self.create_sequences(test_processed, mode='predict')
        
        if len(X) == 0:
            print("예측할 데이터 없음")
            return np.zeros((0, self.pred_length))
        
        # 예측
        has_sales_prob = self.zero_classifier.predict_proba(X)[:, 1]
        sales_pred = self.main_model.predict(X)
        
        # Zero-inflation 적용
        final_pred = sales_pred * (has_sales_prob > 0.3).reshape(-1, 1)
        
        # 음수 제거
        final_pred = np.maximum(final_pred, 0)
        
        return final_pred, metadata
    
    def create_submission(self, test_files):
        """제출 파일 생성"""
        submission = pd.read_csv('sample_submission.csv')
        
        for test_idx, test_file in enumerate(test_files):
            print(f"처리: {test_file}")
            
            # 테스트 데이터 로드
            test_df = pd.read_csv(test_file)
            test_df['date'] = pd.to_datetime(test_df['영업일자'])
            test_df[['store', 'menu']] = test_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
            test_df['sales'] = test_df['매출수량']
            
            # 예측
            predictions, metadata = self.predict(test_df)
            
            # 제출 파일에 매핑
            test_case = f"TEST_{test_idx:02d}"
            self._map_to_submission(submission, test_case, predictions, metadata)
        
        return submission
    
    def _map_to_submission(self, submission, test_case, predictions, metadata):
        """예측 결과를 제출 파일에 매핑"""
        test_rows = submission[submission['영업일자'].str.contains(test_case, na=False)].index.tolist()
        
        if len(test_rows) != 7:
            return
        
        for i, meta in enumerate(metadata):
            if i >= len(predictions):
                break
                
            store = meta['store']
            menu = meta['menu']
            target_col = f"{store}_{menu}"
            
            # 정확한 컬럼명 또는 유사한 컬럼 찾기
            matching_cols = [col for col in submission.columns if col == target_col]
            if not matching_cols:
                matching_cols = [col for col in submission.columns 
                               if col.startswith(f"{store}_") and menu in col]
            
            # 예측값 할당
            for day_idx, row_idx in enumerate(test_rows):
                if day_idx < predictions.shape[1]:
                    pred_value = max(0, predictions[i, day_idx])
                    
                    for col in matching_cols:
                        submission.loc[row_idx, col] = pred_value

# 사용 예시
def run_improved_pipeline():
    """개선된 파이프라인 실행"""
    
    # 데이터 로드
    train_df = pd.read_csv('./train/train.csv')
    train_df['date'] = pd.to_datetime(train_df['영업일자'])
    train_df[['store', 'menu']] = train_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
    train_df['sales'] = train_df['매출수량']
    
    # 모델 학습
    model = ImprovedForecastingSystem()
    model.fit(train_df)
    
    # 제출 파일 생성
    import glob
    test_files = sorted(glob.glob('TEST_*.csv'))
    submission = model.create_submission(test_files)
    
    # 저장
    submission.to_csv('improved_submission.csv', index=False)
    print("개선된 제출 파일 생성 완료")
    
    return submission

# 실행
submission = run_improved_pipeline()

개선된 모델 학습 시작...
학습 시퀀스: (96114, 40), 타겟: (96114, 7)
모델 학습 완료
처리: TEST_00.csv
처리: TEST_01.csv
처리: TEST_02.csv
처리: TEST_03.csv
처리: TEST_04.csv
처리: TEST_05.csv
처리: TEST_06.csv
처리: TEST_07.csv
처리: TEST_08.csv
처리: TEST_09.csv
개선된 제출 파일 생성 완료
